In [ ]:
import sys, importlib, glob, cv2, ultralytics
# local = "D:\\Snowpole Detection\\ultralytics4channel-0a38736761a770f7f7dd80064e20b2d9624eda5b"
# if local not in sys.path: sys.path.insert(0, local)
# # Unload any previously loaded ultralytics modules",
# for m in [k for k in list(sys.modules) if k.split('.')[0] == 'ultralytics']:
#     sys.modules.pop(m, None)
# import ultralytics, importlib
# importlib.reload(ultralytics)
# print('ultralytics loaded from:', ultralytics.__file__)

# Quick loader test",
from ultralytics.data.loaders import LoadImagesAndVideos
dl = LoadImagesAndVideos("SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\\SnowPole_Detection_Dataset\\4ch_rgb_range\\images\\train", batch=1)
paths, imgs, info = next(iter(dl))
print('loader returned:', len(imgs), type(imgs[0]), getattr(imgs[0], 'shape', None))

# If loader still returns 3 channels, replace loader read with IMREAD_UNCHANGED (monkeypatch) and re-test",
if imgs and (getattr(imgs[0], 'shape', (None,))[2] == 3):
    import ultralytics.data.loaders as _ld
    def _new_next(self):
        paths, imgs, info = [], [], []
        while len(imgs) < self.bs:
            if self.count >= self.nf:
                if len(imgs) > 0: return paths, imgs, info
                raise StopIteration
            path = self.files[self.count]
            self.count += 1
            im0 = cv2.imread(path, cv2.IMREAD_UNCHANGED)
            paths.append(path); imgs.append(im0); info.append(f'image {self.count}/{self.nf} {path}: ')
        return paths, imgs, info
    _ld.LoadImagesAndVideos.__next__ = _new_next
    dl = LoadImagesAndVideos("SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\\SnowPole_Detection_Dataset\\4ch_rgb_range\\images\\train", batch=1)
    paths, imgs, info = next(iter(dl))
    print('after monkeypatch loader returned:', getattr(imgs[0], 'shape', None))

ultralytics loaded from: d:\Snowpole Detection\cuda-env\Lib\site-packages\ultralytics\__init__.py
loader returned: 1 <class 'numpy.ndarray'> (1024, 1024, 3)
after monkeypatch loader returned: (1024, 1024, 4)


In [2]:
from ultralytics.data.loaders import LoadImagesAndVideos
dl = LoadImagesAndVideos(r"SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\SnowPole_Detection_Dataset\4ch_rgb_range\images\train", batch=1)
paths, imgs, info = next(iter(dl))
print("loader returned:", len(imgs), type(imgs[0]), getattr(imgs[0], "shape", None))

KeyboardInterrupt: 

In [2]:
import sys
sys.path.insert(0, "/D:\\Snowpole Detection\\ultralytics4channel-0a38736761a770f7f7dd80064e20b2d9624eda5b")  # parent of the ultralytics package

import ultralytics
from ultralytics import YOLO

print("Ultralytics module file:", ultralytics.__file__)


Ultralytics module file: d:\Snowpole Detection\cuda-env\Lib\site-packages\ultralytics\__init__.py


In [3]:
import torch
import cv2
import shutil
import yaml
import glob
import numpy as np
from pathlib import Path
from ultralytics import YOLO

In [ ]:
# ===================== PATHS =====================
ROOT = Path(
    "D:\\Snowpole Detection\\SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions"
    "\\SnowPole_Detection_Dataset"
)

COMB_ROOT   = ROOT / "combined_color"
RANGE_ROOT  = ROOT / "range"
DUAL_ROOT   = ROOT / "4ch_rgb_range"
LABELS_ROOT = ROOT / "labels"

(DUAL_ROOT / "images").mkdir(parents=True, exist_ok=True)
(DUAL_ROOT / "labels").mkdir(parents=True, exist_ok=True)

# ===================== RANGE PREPROCESS =====================
def preprocess_range(img):
    img = cv2.resize(img, (512, 514))
    img = cv2.equalizeHist(img)
    img = img.astype(np.float32) / 255.0
    return img

In [5]:
# ===================== BUILD 4-CHANNEL DATA =====================
def make_split(split):
    comb_dir  = COMB_ROOT / split
    range_dir = RANGE_ROOT / split
    dual_dir  = DUAL_ROOT / "images" / split
    lbl_dir   = DUAL_ROOT / "labels" / split

    dual_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    # copy labels
    for f in (LABELS_ROOT / split).glob("*.txt"):
        shutil.copy2(f, lbl_dir / f.name)

    imgs = list(comb_dir.glob("*.*"))
    print(f"{split}: {len(imgs)} images")

    for img_path in imgs:
        rgb = cv2.imread(str(img_path))
        if rgb is None:
            continue

        stem = img_path.stem

        rpath = None
        for ext in [".png", ".jpg", ".jpeg"]:
            candidate = range_dir / f"{stem}{ext}"
            if candidate.exists():
                rpath = candidate
                break

        if rpath is None:
            print("Missing range:", stem)
            continue

        rimg = cv2.imread(str(rpath), cv2.IMREAD_GRAYSCALE)
        rimg = preprocess_range(rimg)

        rgb = cv2.resize(rgb, (1024, 1024))
        rgb = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)
        rgba = np.dstack([rgb, rimg])

        outpath = dual_dir / f"{stem}.png"
        rgba = (rgba * 255).astype(np.uint8) if rgba.dtype != np.uint8 else rgba
        cv2.imwrite(str(outpath), rgba)


# ===================== RUN PREPROCESS =====================
for split in ["train", "valid", "test"]:
    make_split(split)

# ===================== SANITY CHECK =====================
for split in ["train", "valid", "test"]:
    paths = glob.glob(str(DUAL_ROOT / f"images/{split}/*.png"))
    for p in paths:
        img = cv2.imread(p, cv2.IMREAD_UNCHANGED)
        if img is None or img.shape[2] != 4:
            print("BAD IMAGE:", p)

train: 1367 images
valid: 390 images
test: 197 images


In [6]:
# ===================== DATA YAML =====================
data_yaml = DUAL_ROOT / "data.yaml"

cfg = {
    "path": str(DUAL_ROOT),
    "train": "images/train",
    "val":   "images/valid",
    "test":  "images/test",
    "names": ["snow_pole"],
    "nc": 1,
    "channels": 4
}

with open(data_yaml, "w") as f:
    yaml.safe_dump(cfg, f)

print(data_yaml.read_text())

channels: 4
names:
- snow_pole
nc: 1
path: D:\Snowpole Detection\SnowPole Detection A Comprehensive Dataset for Detection
  and Localization Using LiDAR Imaging in Nordic Winter Conditions\SnowPole_Detection_Dataset\4ch_rgb_range
test: images/test
train: images/train
val: images/valid



In [7]:
# ===================== YOLO TRAINING (FROM 1st CODE INTENT) =====================
model = YOLO("yolov8n.yaml")  # or yolov8s.pt if VRAM allows

model.train(
    data=r"D:\\Snowpole Detection\\SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\\SnowPole_Detection_Dataset\\4ch_rgb_range\\data.yaml",
    imgsz=1024,
    epochs=100,
    batch=2,                 # tune based on GPU
    device=0,                # or "cpu"
    project="SnowPole_4ch",
    name="yolo_rgb_range",
    amp=False,
    bgr=0.0,
    workers=0
    
)

New https://pypi.org/project/ultralytics/8.3.241 available  Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.2.5  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: task=detect, mode=train, model=yolov8n.yaml, data=D:\\Snowpole Detection\\SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\\SnowPole_Detection_Dataset\\4ch_rgb_range\\data.yaml, epochs=100, time=None, patience=100, batch=2, imgsz=1024, save=True, save_period=-1, cache=False, device=0, workers=0, project=SnowPole_4ch, name=yolo_rgb_range13, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=False, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=3

d:\Snowpole Detection\cuda-env\Lib\site-packages\ultralytics\engine\trainer.py:262: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(enabled=self.amp)
train: Scanning D:\Snowpole Detection\SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\SnowPole_Detection_Dataset\4ch_rgb_range\labels\train.cache... 1367 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1367/1367 [00:00<?, ?it/s]
val: Scanning D:\Snowpole Detection\SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\SnowPole_Detection_Dataset\4ch_rgb_range\labels\valid.cache... 390 images, 0 backgrounds, 0 corrupt: 100%|██████████| 390/390 [00:00<?, ?it/s]


Plotting labels to SnowPole_4ch\yolo_rgb_range13\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Image sizes 1024 train, 1024 val
Using 0 dataloader workers
Logging results to SnowPole_4ch\yolo_rgb_range13
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      1/100      1.36G      3.006       20.4      1.448          4       1024: 100%|██████████| 684/684 [05:03<00:00,  2.25it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 98/98 [01:00<00:00,  1.63it/s]

                   all        390        789          0          0          0          0


...

                   all        390        789      0.719       0.62      0.697      0.273



100 epochs completed in 9.110 hours.
Optimizer stripped from SnowPole_4ch\yolo_rgb_range13\weights\last.pt, 6.3MB
Optimizer stripped from SnowPole_4ch\yolo_rgb_range13\weights\best.pt, 6.3MB

Validating SnowPole_4ch\yolo_rgb_range13\weights\best.pt...
Ultralytics YOLOv8.2.5  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
YOLOv8n summary (fused): 168 layers, 3005987 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 98/98 [00:35<00:00,  2.73it/s]


                   all        390        789      0.733      0.599      0.712      0.288
Speed: 0.9ms preprocess, 52.2ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to SnowPole_4ch\yolo_rgb_range13


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000026384950210>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [1]:
import torch, cv2, shutil, yaml
# from pathlib import Path
# import numpy as np
# from ultralytics.nn.modules import Conv
# from ultralytics import YOLO
# import cv2, glob

In [2]:
print("CUDA:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA: True
NVIDIA GeForce GTX 1650


In [10]:
ROOT = Path("D:\\Snowpole Detection\\SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\\SnowPole_Detection_Dataset")

COMB_ROOT  = ROOT / "combined_color"
RANGE_ROOT = ROOT / "range"
DUAL_ROOT  = ROOT / "4ch_rgb_range"
LABELS_ROOT = ROOT / "labels"

DUAL_ROOT.mkdir(parents=True, exist_ok=True)


In [11]:
def preprocess_range(img):
    img = cv2.resize(img, (1024, 1024))
    img = cv2.equalizeHist(img)
    img = img.astype(np.float32) / 255.0
    return img


In [12]:
def make_split(split):
    comb_dir  = COMB_ROOT  / split
    range_dir = RANGE_ROOT / split
    dual_dir  = DUAL_ROOT  / "images" / split
    lbl_dir   = DUAL_ROOT  / "labels" / split

    dual_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    # copy labels
    for f in (LABELS_ROOT / split).glob("*.txt"):
        shutil.copy2(f, lbl_dir/f.name)

    imgs = list(comb_dir.glob("*.*"))
    print(split, len(imgs), "images")

    for img_path in imgs:

        rgb = cv2.imread(str(img_path))      # BGR
        if rgb is None:
            continue

        stem = img_path.stem

        candidates = [
            range_dir / f"{stem}.png",
            range_dir / f"{stem}.jpg",
            range_dir / f"{stem}.jpeg",
        ]

        rpath = None
        for c in candidates:
            if c.exists():
                rpath = c
                break

        if rpath is None:
            print("missing range:", stem)
            continue
        # print("rgb:", img_path.exists(), "range:", rpath.exists())


        rimg = cv2.imread(str(rpath), cv2.IMREAD_GRAYSCALE)
        rimg = preprocess_range(rimg)

        # resize RGB too (keep same size)
        rgb = cv2.resize(rgb, (1024, 1024))

        rgba = np.dstack([rgb, rimg])
        outpath = dual_dir / f"{stem}.png"
        print("saving:", outpath)
        cv2.imwrite(str(outpath), rgba)


        # cv2.imwrite(str(dual_dir / f"{stem}.png"), rgba)


In [13]:


for split in ['train', 'val', 'test']:
    paths = glob.glob(str(DUAL_ROOT/f'images/{split}/*.png'))
    for p in paths:
        img = cv2.imread(p, cv2.IMREAD_UNCHANGED)
        if img.shape[2] != 4:
            print("BAD IMAGE:", p, img.shape)



In [14]:
for split in ['train', 'val', 'test']:
    make_split(split)

train 1367 images
saving: D:\Snowpole Detection\SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\SnowPole_Detection_Dataset\4ch_rgb_range\images\train\image_1.png
saving: D:\Snowpole Detection\SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\SnowPole_Detection_Dataset\4ch_rgb_range\images\train\image_10.png
saving: D:\Snowpole Detection\SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\SnowPole_Detection_Dataset\4ch_rgb_range\images\train\image_100.png
saving: D:\Snowpole Detection\SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\SnowPole_Detection_Dataset\4ch_rgb_range\images\train\image_1000.png
saving: D:\Snowpole Detection\SnowPole Detection A Comprehensive Dataset for Detection and Localization 

In [15]:
data_yaml = DUAL_ROOT / "data.yaml"

cfg = {
    "path": str(DUAL_ROOT),
    "train": "images/train",
    "val":   "images/val",
    "test":  "images/test",
    "names": ["snow_pole"],
    "nc": 1,
    "channels": 4
}

with open(data_yaml, "w") as f:
    yaml.safe_dump(cfg, f)

print(data_yaml.read_text())


channels: 4
names:
- snow_pole
nc: 1
path: D:\Snowpole Detection\SnowPole Detection A Comprehensive Dataset for Detection
  and Localization Using LiDAR Imaging in Nordic Winter Conditions\SnowPole_Detection_Dataset\4ch_rgb_range
test: images/test
train: images/train
val: images/val



In [16]:
# ===================== YOLO TRAINING (FROM 1st CODE INTENT) =====================
model = YOLO("yolov8n.yaml")  # or yolov8s.pt if VRAM allows

model.train(
    data=str(data_yaml),
    imgsz=1024,
    epochs=100,
    batch=4,                 # tune based on GPU
    device=0,                # or "cpu"
    project="SnowPole_4ch",
    name="yolo_rgb_range",
    amp=False,
    bgr=0.0
    
)

New https://pypi.org/project/ultralytics/8.3.241 available  Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.2.5  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)


RuntimeError: CUDA error: an illegal memory access was encountered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
import torch
import torch.nn as nn
from ultralytics import YOLO

# --------------------------------------------------------
# Load pretrained YOLOv9
# --------------------------------------------------------
model = YOLO("yolov9t.pt")

net = model.model      # inner model
stem = net.model[0]    # first conv block
old_conv = stem.conv

print("Original:", old_conv)

# --------------------------------------------------------
# Build new Conv that supports 4 channels
# --------------------------------------------------------
new_conv = nn.Conv2d(
    in_channels=4,
    out_channels=old_conv.out_channels,
    kernel_size=old_conv.kernel_size,
    stride=old_conv.stride,
    padding=old_conv.padding,
    bias=old_conv.bias is not None
)

with torch.no_grad():
    new_conv.weight[:, :3] = old_conv.weight
    new_conv.weight[:, 3:] = torch.zeros_like(old_conv.weight[:, :1])

    if old_conv.bias is not None:
        new_conv.bias.copy_(old_conv.bias)

# --------------------------------------------------------
# Replace the conv safely
# --------------------------------------------------------
stem.conv = new_conv
net.model[0] = stem

print("Patched:", net.model[0].conv)

# --------------------------------------------------------
# Save modified model
# --------------------------------------------------------
model.save("yolov9_4ch.pt")
print("Saved modified model.")


Original: Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
Patched: Conv2d(4, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
Saved modified model.


In [ ]:
import torch
import torch.nn as nn
import cv2
import shutil
import yaml
import numpy as np
from pathlib import Path
from ultralytics import YOLO

# ------------------- Setup -------------------
print("CUDA:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

ROOT = Path(r"SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\SnowPole_Detection_Dataset")
COMB_ROOT  = ROOT / "combined_color"
RANGE_ROOT = ROOT / "range"
DUAL_ROOT  = ROOT / "4ch_rgb_range"
LABELS_ROOT = ROOT / "labels"

DUAL_ROOT.mkdir(parents=True, exist_ok=True)

# ------------------- Helpers -------------------
def preprocess_range(img):
    img = cv2.resize(img, (1024, 1024))
    img = cv2.equalizeHist(img)
    img = img.astype(np.float32) / 255.0
    return img

def make_split(split):
    comb_dir  = COMB_ROOT  / split
    range_dir = RANGE_ROOT / split
    dual_dir  = DUAL_ROOT  / "images" / split
    lbl_dir   = DUAL_ROOT  / "labels" / split

    dual_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    # copy labels
    for f in (LABELS_ROOT / split).glob("*.txt"):
        shutil.copy2(f, lbl_dir/f.name)

    imgs = list(comb_dir.glob("*.*"))
    print(split, len(imgs), "images")

    for img_path in imgs:
        rgb = cv2.imread(str(img_path))
        if rgb is None:
            continue

        stem = img_path.stem

        # find range image
        rpath = None
        for ext in [".png", ".jpg", ".jpeg"]:
            candidate = range_dir / f"{stem}{ext}"
            if candidate.exists():
                rpath = candidate
                break

        if rpath is None:
            print("missing range:", stem)
            continue

        rimg = cv2.imread(str(rpath), cv2.IMREAD_GRAYSCALE)
        rimg = preprocess_range(rimg)
        rgb = cv2.resize(rgb, (1024, 1024))

        # combine into 4-channel
        rgba = np.dstack([rgb, (rimg*255).astype(np.uint8)])
        outpath = dual_dir / f"{stem}.png"
        cv2.imwrite(str(outpath), rgba)

# ------------------- Preprocess all splits -------------------
for split in ['train', 'valid', 'test']:
    make_split(split)

# ------------------- YAML -------------------
data_yaml = DUAL_ROOT / "data.yaml"
cfg = {
    "path": str(DUAL_ROOT),
    "train": "images/train",
    "val": "images/valid",
    "test": "images/test",
    "names": ["snow_pole"],
    "nc": 1,
    "channels": 4
}
with open(data_yaml, "w") as f:
    yaml.safe_dump(cfg, f)

print(data_yaml.read_text())

# ------------------- Patch YOLO model -------------------
model = YOLO("yolov9t.pt")
net = model.model
stem = net.model[0]  # first Conv block
old_conv = stem.conv

# create new Conv with 4 input channels
new_conv = nn.Conv2d(
    in_channels=4,
    out_channels=old_conv.out_channels,
    kernel_size=old_conv.kernel_size,
    stride=old_conv.stride,
    padding=old_conv.padding,
    bias=old_conv.bias is not None
)
with torch.no_grad():
    new_conv.weight[:, :3] = old_conv.weight  # copy RGB weights
    new_conv.weight[:, 3:] = torch.zeros_like(old_conv.weight[:, :1])
    if old_conv.bias is not None:
        new_conv.bias.copy_(old_conv.bias)

stem.conv = new_conv
net.model[0] = stem
model.save("yolov9_4ch.pt")
print("Saved patched 4-channel model.")


# ------------------- Train -------------------
data_dict = {
    "channels": 4,
    "names": ["snow_pole"],
    "nc": 1,
    "train": str(DUAL_ROOT / "images/train"),
    "val": str(DUAL_ROOT / "images/valid"),
    "test": str(DUAL_ROOT / "images/test")
}

model = YOLO("yolov9_4ch.pt")
results = model.train(
    data=data_dict,  # pass dict directly
    imgsz=1024,
    epochs=50,
    batch=4,
    device=0,
    workers=2
)



CUDA: True
NVIDIA GeForce GTX 1650
train 1367 images
valid 390 images
test 197 images
channels: 4
names:
- snow_pole
nc: 1
path: SnowPole Detection A Comprehensive Dataset for Detection and Localization Using
  LiDAR Imaging in Nordic Winter Conditions\SnowPole_Detection_Dataset\4ch_rgb_range
test: images/test
train: images/train
val: images/valid

Saved patched 4-channel model.


ModuleNotFoundError: No module named 'ultralytics.data.dataloaders'

In [ ]:
import yaml
from pathlib import Path
from ultralytics import YOLO

# Define dataset
data_dict = {
    "channels": 4,
    "names": ["snow_pole"],
    "nc": 1,
    "train": r"SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\SnowPole_Detection_Dataset\4ch_rgb_range\images\train",
    "val": r"SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\SnowPole_Detection_Dataset\4ch_rgb_range\images\valid",
    "test": r"SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\SnowPole_Detection_Dataset\4ch_rgb_range\images\test"
}

# Write to temporary YAML
yaml_path = Path("temp_4ch_data.yaml")
with open(yaml_path, "w") as f:
    yaml.dump(data_dict, f)

# Load model and train
model = YOLO("yolov9_4ch.pt")
results = model.train(
    data=str(yaml_path),
    imgsz=1024,
    epochs=50,
    batch=4,
    device=0,
    workers=2
)


Ultralytics 8.3.233  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=temp_4ch_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov9_4ch.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train15, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, po

RuntimeError: Given groups=1, weight of size [16, 4, 3, 3], expected input[4, 3, 1024, 1024] to have 4 channels, but got 3 channels instead

In [49]:
import torch
from torch.utils.data import Dataset, DataLoader
import cv2, numpy as np
from ultralytics import YOLO
from pathlib import Path

class FourChannelDataset(Dataset):
    def __init__(self, img_dir, label_dir, img_size=1024):
        self.img_dir = Path(img_dir)
        self.label_dir = Path(label_dir)
        self.img_paths = list(self.img_dir.glob("*.png"))
        self.img_size = img_size

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        label_path = self.label_dir / f"{img_path.stem}.txt"

        img = cv2.imread(str(img_path), cv2.IMREAD_UNCHANGED)
        img = cv2.resize(img, (self.img_size, self.img_size))

        img = img.astype(np.float32) / 255.0
        img = np.transpose(img, (2,0,1))

        labels = []
        if label_path.exists():
            with open(label_path) as f:
                for line in f:
                    labels.append([float(x) for x in line.split()])
        labels = torch.tensor(labels) if labels else torch.zeros((0,5))

        return torch.tensor(img), labels


def collate_fn(batch):
    imgs, labels = zip(*batch)
    imgs = torch.stack(imgs)
    return imgs, labels

# Paths
train_dataset = FourChannelDataset("SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\\SnowPole_Detection_Dataset\\4ch_rgb_range\\images\\train", "SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\\SnowPole_Detection_Dataset\\4ch_rgb_range\\labels\\train")
val_dataset   = FourChannelDataset("SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\\SnowPole_Detection_Dataset\\4ch_rgb_range\\images\\valid", "SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\\SnowPole_Detection_Dataset\\4ch_rgb_range\\labels\\valid")

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn
)

model = YOLO("yolov9_4ch.pt")

for imgs, labels in train_loader:
    print(imgs.shape)    # [B,4,1024,1024]
    print(len(labels))   # batch size
    break


torch.Size([4, 4, 1024, 1024])
4


In [52]:
import torch

model = YOLO("yolov8n.pt").model  # get raw nn.Module

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
model.train()

for epoch in range(30):
    for imgs, labels in train_loader:
        preds = model(imgs)
        loss = preds[0]  # YOLO returns tuple
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()


RuntimeError: Given groups=1, weight of size [16, 3, 3, 3], expected input[4, 4, 1024, 1024] to have 3 channels, but got 4 channels instead

In [ ]:
metrics = model.val(data=str(data_yaml), device=0, save_json=True)
print(metrics)


In [ ]:
P = metrics.results_dict['metrics/precision']
R = metrics.results_dict['metrics/recall']
F1 = 2*P*R/(P+R)
print("F1 Score:", F1)

In [ ]:
miou = metrics.results_dict["metrics/segment/miou"] \
    if "metrics/segment/miou" in metrics.results_dict else None

print("mIoU:", miou)

##DOWNLOADED CODE

In [1]:
import torch
from ultralytics import YOLO
from pathlib import Path
import shutil
import cv2
import numpy as np
import yaml

In [ ]:
COMB_ROOT  = Path("SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions\\SnowPole_Detection_Dataset\\combined_color\\train")

# 1-channel range-normalized images
RANGE_ROOT = Path("/content/drive/MyDrive/SnowPole_Detection_Dataset/range_normalized")

# New 4-channel dual-input dataset
DUAL_ROOT  = Path("/content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range")
DUAL_ROOT.mkdir(parents=True, exist_ok=True)

print("COMB_ROOT :", COMB_ROOT)
print("RANGE_ROOT:", RANGE_ROOT)
print("DUAL_ROOT :", DUAL_ROOT)

In [ ]:
def make_dual_split(split: str):
    comb_img_dir   = COMB_ROOT  / "images" / split
    range_img_dir  = RANGE_ROOT / "images" / split
    dual_img_dir   = DUAL_ROOT  / "images" / split
    dual_lbl_dir   = Path("/content/drive/MyDrive/SnowPole_Detection_Dataset/labels") / split


    dual_img_dir.mkdir(parents=True, exist_ok=True)
    dual_lbl_dir.mkdir(parents=True, exist_ok=True)

    # copy labels for COMB_ROOT but they r same(assumes identical gt for both modalities)
    src_lbl_dir = dual_lbl_dir
    for lbl in src_lbl_dir.glob("*.txt"):
        shutil.copy2(lbl, dual_lbl_dir / lbl.name)

    img_files = sorted(comb_img_dir.glob("*.*"))
    print(f"[{split}] Found {len(img_files)} images")

    for comb_path in img_files:
        stem = comb_path.stem

        # read comb RGB (3ch, BGR)
        comb = cv2.imread(str(comb_path), cv2.IMREAD_COLOR)
        if comb is None:
            print("Could not read comb image:", comb_path)
            continue

        # read range image (1ch)
        range_path = range_img_dir / f"{stem}.png"
        if not range_path.exists():
            # try jpg as fallback
            range_path = range_img_dir / f"{stem}.jpg"

        # DO NOT READ AS GRAYSCALE BUT TAKE ONE OF THE CHANNEL from there which is it's same stuff
        range_img = cv2.imread(str(range_path), cv2.IMREAD_GRAYSCALE)


        # ensure same size
        if comb.shape[:2] != range_img.shape[:2]:
            range_img = cv2.resize(range_img, (comb.shape[1], comb.shape[0]))

        # stack into 4-channel: B,G,R,range
        rgba = np.dstack([comb, range_img])

        out_path = dual_img_dir / f"{stem}.png"
        cv2.imwrite(str(out_path), rgba)

# let em cook
for split in ["train", "val", "test"]:
    make_dual_split(split)


In [ ]:
ORIG_DATA_YAML = COMB_ROOT / "data.yaml"
DUAL_DATA_YAML = DUAL_ROOT / "data.yaml"

with open(ORIG_DATA_YAML, "r") as f:
    cfg = yaml.safe_load(f)

base = DUAL_ROOT

def make_rel(p):
    # p might be absolute or relative – we point to new dual root
    p = Path(p)
    return str((base / "images" / p.name).parent)  # keep split names

# If your original yaml used explicit paths, you can instead do:
# cfg["path"]  = str(DUAL_ROOT)
cfg["path"]  = str(DUAL_ROOT)
cfg["train"] = "images/train"
cfg["val"]   = "images/val"
cfg["test"]  = "images/test"
cfg["channels"] = 4          # tell YOLO this is 4-channel data with the RGB-Alpha

with open(DUAL_DATA_YAML, "w") as f:
    yaml.safe_dump(cfg, f)

print(DUAL_DATA_YAML.read_text())

In [ ]:
%yolo train \
%  model="yolov9t_dual_4ch.pt" \
%  data="{DUAL_DATA_YAML}" \
%  epochs=400 \ # 400 and training it untill no improvement is seen, and letting it train untill stop loss, it may finish earlier at 300 or 250. in the case,
%                # it runs till the full 400, use the yolo resume command and extend till 500 or 450 untill it reacher early stop
%  imgsz=1024 \
%  device=0 \
%  batch=16 \
%  name="dual_comb_rgb_plus_range_9t" \
%  project="dual_comb_range_experiments"
